# P1-E09.1 — CAUEEG Benchmark Stability, Confound and Error-Analysis Lock

**Paper 1, Experiment 09.1 — diagnostic successor to the passed P1-E08. benchmark lock**

## Goal

Determine whether the locked conventional benchmarks are stable, whether age or data-quantity variables may explain their performance, and where the models fail. This notebook does **not** conduct another model search.

It performs:

1. ten repeated stratified five-fold evaluations on the combined training + validation development cohort, using the exact P1-E08.1 representation, model family and hyperparameters;
2. age-only benchmarking;
3. fold-wise age-residualized EEG sensitivity analysis;
4. equal-data sensitivity analysis restricted to recordings with exactly 12 selected epochs;
5. feature-importance stability analysis;
6. recording-level associations between out-of-fold errors and age, retained duration, selected epochs, and automatic-QA acceptance;
7. descriptive confidence, calibration and error analysis of the already-locked official test predictions; and
8. a final stability/confound audit lock.

The official test results are descriptive only and cannot change the benchmark or any preprocessing decision.

Run all cells in order. Continue only if the final status is:

> **P1-E09.1 PASSED — STABILITY, CONFOUND AND ERROR AUDIT LOCKED**

## Context & Methods

### Interpretation rules

- **Locked EEG:** exact P1-E08.1 estimator and representation.
- **Age only:** fixed class-weighted logistic regression using age as the sole predictor.
- **Age-residualized EEG:** within every fold, linear age effects are fitted on the fold-training feature matrix and removed from both fold-training and fold-validation data before fitting the locked estimator.
- **Equal-12-epoch sensitivity:** repeats the locked evaluation only among recordings contributing exactly 12 epochs.
- No scenario is allowed to replace the P1-E08. model based on these results.
- Repeated cross-validation measures development-cohort stability; it is not an additional independent external validation.
- Sex cannot be assessed because it was removed from the released CAUEEG dataset.
- Participant identity remains unverified, so all resampling remains recording-level.

## Setup

### 1. Paths, dependencies and diagnostic design

In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix)
import joblib

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E07_ROOT=PAPER_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
E08_ROOT=PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
OUTPUT_ROOT=PAPER_ROOT/'p1_e09_1_benchmark_stability_confound_error_audit'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'
for path in (OUTPUT_ROOT,TABLES,FIGURES): path.mkdir(parents=True,exist_ok=True)

E08_RUN_RECORD=E08_ROOT/'run_record.json'
FEATURE_MATRIX=E08_ROOT/'features'/'02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
FEATURE_METADATA=E08_ROOT/'tables'/'05_LOCKED_RECORDING_FEATURE_METADATA_WITH_LABELS.csv'
SELECTED_CANDIDATES=E08_ROOT/'tables'/'08_LOCKED_SELECTED_CANDIDATES_BEFORE_TEST.csv'
TEST_RESULTS=E08_ROOT/'tables'/'09_FINAL_OFFICIAL_TEST_RESULTS.csv'
TEST_PREDICTIONS=E08_ROOT/'tables'/'11_OFFICIAL_TEST_RECORDING_PREDICTIONS.csv'
E07_RECORDINGS=E07_ROOT/'tables'/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'

TASKS={'three_class':['Normal','MCI','AD'],'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD']}
REPEATS=10; FOLDS=5; SEED=20260828

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,
                     'axes.titlesize':10,'axes.labelsize':9,'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='whitegrid',context='paper')

def sha256_file(path,block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')
def metrics(y_true,y_pred):
    return {'accuracy':accuracy_score(y_true,y_pred),
            'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro'),
            'mcc':matthews_corrcoef(y_true,y_pred)}

print('Experiment: P1-E09.1')
print('Input:',E08_ROOT)
print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate P1-E08.1 and reconstruct the locked analysis objects

In [ ]:
required=[E08_RUN_RECORD,FEATURE_MATRIX,FEATURE_METADATA,SELECTED_CANDIDATES,
          TEST_RESULTS,TEST_PREDICTIONS,E07_RECORDINGS]
model_paths={task:E08_ROOT/'models'/f'{task}_LOCKED_NONSPIKING_BENCHMARK.joblib' for task in TASKS}
required.extend(model_paths.values())
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Required locked inputs are missing:\n'+'\n'.join(missing))

e08_run=json.loads(E08_RUN_RECORD.read_text(encoding='utf-8'))
with np.load(FEATURE_MATRIX,allow_pickle=False) as payload:
    X=np.asarray(payload['X'],dtype=float)
    feature_ids=payload['recording_id'].astype(str)
    feature_names=payload['feature_names'].astype(str)
metadata=pd.read_csv(FEATURE_METADATA,dtype={'recording_id':str,'serial':str})
selected=pd.read_csv(SELECTED_CANDIDATES)
locked_test_results=pd.read_csv(TEST_RESULTS)
locked_test_predictions=pd.read_csv(TEST_PREDICTIONS,dtype={'recording_id':str,'serial':str})
e07=pd.read_csv(E07_RECORDINGS,dtype={'recording_id':str,'serial':str})
for frame in (metadata,locked_test_predictions,e07):
    for column in ('recording_id','serial'):
        if column in frame: frame[column]=frame[column].astype(str).str.strip().str.zfill(5)

if metadata.recording_id.tolist()!=feature_ids.tolist(): raise ValueError('Feature-row order does not match metadata')
qa_columns=['recording_id','selected_epochs','accepted_fraction','retained_seconds','candidate_epochs','accepted_epochs']
qa_columns=[column for column in qa_columns if column in e07.columns]
metadata=metadata.merge(e07[qa_columns],on='recording_id',how='left',validate='one_to_one')
metadata['age']=pd.to_numeric(metadata.age,errors='coerce')
age_missing=float(metadata.age.isna().mean())
artifacts={task:joblib.load(path) for task,path in model_paths.items()}

input_checks=pd.DataFrame([
 {'check':'P1-E08.1 passed','pass':str(e08_run.get('status','')).startswith('P1-E08.1 PASSED'),'detail':e08_run.get('status','')},
 {'check':'feature matrix authenticated','pass':sha256_file(FEATURE_MATRIX)==e08_run.get('feature_matrix_sha256'),'detail':sha256_file(FEATURE_MATRIX)},
 {'check':'feature metadata authenticated','pass':sha256_file(FEATURE_METADATA)==e08_run.get('feature_metadata_sha256'),'detail':sha256_file(FEATURE_METADATA)},
 {'check':'selected candidates authenticated','pass':sha256_file(SELECTED_CANDIDATES)==e08_run.get('selected_candidates_sha256'),'detail':sha256_file(SELECTED_CANDIDATES)},
 {'check':'official test results authenticated','pass':sha256_file(TEST_RESULTS)==e08_run.get('official_test_results_sha256'),'detail':sha256_file(TEST_RESULTS)},
 {'check':'914 finite feature rows','pass':X.shape[0]==914 and np.isfinite(X).all(),'detail':X.shape},
 {'check':'three locked model artifacts available','pass':set(artifacts)==set(TASKS),'detail':list(artifacts)},
 {'check':'age sufficiently complete for sensitivity analysis','pass':age_missing<=0.10,'detail':age_missing},
 {'check':'analysis remains recording-level','pass':metadata.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':metadata.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~metadata.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':metadata.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(input_checks,'01_input_authentication_checks'); display(input_checks)
if not input_checks['pass'].all(): raise AssertionError('STOP — P1-E08.1 INPUT AUTHENTICATION FAILED')
print('P1-E08.1 BENCHMARK LOCK AUTHENTICATED')

## Results

### 3. Repeated development-cohort stability and confound sensitivities

In [ ]:
def age_residualize(X_train,X_valid,age_train,age_valid):
    median=float(np.nanmedian(age_train))
    age_train=np.where(np.isfinite(age_train),age_train,median)
    age_valid=np.where(np.isfinite(age_valid),age_valid,median)
    centered=age_train-age_train.mean(); denominator=np.sum(centered**2)+1e-12
    feature_mean=X_train.mean(axis=0)
    slope=np.sum(centered[:,None]*(X_train-feature_mean),axis=0)/denominator
    intercept=feature_mean-slope*age_train.mean()
    return X_train-(intercept+age_train[:,None]*slope), X_valid-(intercept+age_valid[:,None]*slope)

def predict_confidence(estimator,X_use):
    if hasattr(estimator,'predict_proba'):
        probability=estimator.predict_proba(X_use)
        return probability.max(axis=1)
    score=estimator.decision_function(X_use)
    score=np.asarray(score)
    if score.ndim==1: return 1/(1+np.exp(-np.abs(score)))
    shifted=score-score.max(axis=1,keepdims=True); probability=np.exp(shifted)/np.exp(shifted).sum(axis=1,keepdims=True)
    return probability.max(axis=1)

prediction_rows=[]; repeat_metric_rows=[]; importance_rows=[]
for task,classes in TASKS.items():
    artifact=artifacts[task]; columns=np.asarray(artifact['feature_indices'],dtype=int)
    development=metadata.experimental_label.isin(classes) & metadata.official_split.isin(['train','validation'])
    base_indices=np.flatnonzero(development.to_numpy())
    scenarios={'locked_eeg':base_indices,
               'age_only':base_indices,
               'age_residualized_eeg':base_indices,
               'equal_12_epochs':np.flatnonzero((development & metadata.selected_epochs.eq(12)).to_numpy())}
    for scenario,indices in scenarios.items():
        y=metadata.experimental_label.to_numpy()[indices]
        if len(np.unique(y))!=len(classes): raise ValueError(f'{task}/{scenario}: class missing')
        for repeat in range(REPEATS):
            splitter=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED+repeat)
            repeat_true=[]; repeat_pred=[]
            for fold,(local_train,local_valid) in enumerate(splitter.split(indices,y),1):
                train_index=indices[local_train]; valid_index=indices[local_valid]
                y_train=metadata.experimental_label.to_numpy()[train_index]
                y_valid=metadata.experimental_label.to_numpy()[valid_index]
                train_age=metadata.age.to_numpy(float)[train_index]
                valid_age=metadata.age.to_numpy(float)[valid_index]
                if scenario=='age_only':
                    median=float(np.nanmedian(train_age))
                    train_matrix=np.where(np.isfinite(train_age),train_age,median).reshape(-1,1)
                    valid_matrix=np.where(np.isfinite(valid_age),valid_age,median).reshape(-1,1)
                    estimator=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(
                        C=1.0,class_weight='balanced',max_iter=4000,random_state=SEED))])
                else:
                    train_matrix=X[train_index][:,columns]; valid_matrix=X[valid_index][:,columns]
                    if scenario=='age_residualized_eeg':
                        train_matrix,valid_matrix=age_residualize(train_matrix,valid_matrix,train_age,valid_age)
                    estimator=clone(artifact['estimator'])
                estimator.fit(train_matrix,y_train)
                prediction=estimator.predict(valid_matrix); confidence=predict_confidence(estimator,valid_matrix)
                repeat_true.extend(y_valid); repeat_pred.extend(prediction)
                for global_index,true,pred,conf in zip(valid_index,y_valid,prediction,confidence):
                    prediction_rows.append({'task':task,'scenario':scenario,'repeat':repeat,'fold':fold,
                        'recording_id':metadata.recording_id.iloc[global_index],'true_label':true,
                        'predicted_label':pred,'correct':bool(true==pred),'confidence':float(conf)})
                if scenario=='locked_eeg':
                    model=estimator.named_steps['model']
                    if hasattr(model,'coef_'): importance=np.mean(np.abs(model.coef_),axis=0)
                    elif hasattr(model,'feature_importances_'): importance=model.feature_importances_
                    else: importance=np.full(len(columns),np.nan)
                    top=np.argsort(importance)[::-1][:30]
                    for rank,position in enumerate(top,1):
                        importance_rows.append({'task':task,'repeat':repeat,'fold':fold,'rank':rank,
                            'feature':feature_names[columns[position]],'importance':float(importance[position])})
            repeat_metric_rows.append({'task':task,'scenario':scenario,'repeat':repeat,
                                       'n_recordings':len(indices),**metrics(repeat_true,repeat_pred)})
        print(task,scenario,'complete')

oof_predictions=pd.DataFrame(prediction_rows)
repeat_metrics=pd.DataFrame(repeat_metric_rows)
fold_importance=pd.DataFrame(importance_rows)
stability_summary=(repeat_metrics.groupby(['task','scenario']).agg(
    n_recordings=('n_recordings','first'),repeats=('repeat','nunique'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),sd_balanced_accuracy=('balanced_accuracy','std'),
    p025_balanced_accuracy=('balanced_accuracy',lambda x:x.quantile(.025)),
    p975_balanced_accuracy=('balanced_accuracy',lambda x:x.quantile(.975)),
    mean_macro_f1=('macro_f1','mean'),sd_macro_f1=('macro_f1','std'),
    mean_mcc=('mcc','mean'),sd_mcc=('mcc','std')).reset_index())
save_table(repeat_metrics,'02_repeated_development_metrics')
save_table(stability_summary,'03_stability_and_sensitivity_summary')
save_table(oof_predictions,'04_repeated_out_of_fold_predictions')
save_table(fold_importance,'05_fold_feature_importance')
display(stability_summary)

### 4. Quantify confound associations and feature stability

In [ ]:
recording_performance=(oof_predictions.loc[oof_predictions.scenario.eq('locked_eeg')]
    .groupby(['task','recording_id']).agg(oof_accuracy=('correct','mean'),
         mean_confidence=('confidence','mean')).reset_index())
recording_performance=recording_performance.merge(metadata[['recording_id','age','retained_seconds',
    'selected_epochs','accepted_fraction','experimental_label']],on='recording_id',how='left',validate='many_to_one')

association_rows=[]
for task,group in recording_performance.groupby('task'):
    for variable in ['age','retained_seconds','selected_epochs','accepted_fraction']:
        use=group[[variable,'oof_accuracy']].dropna()
        rho,p=spearmanr(use[variable],use.oof_accuracy) if use[variable].nunique()>1 else (np.nan,np.nan)
        association_rows.append({'task':task,'variable':variable,'n':len(use),
                                 'spearman_rho_with_oof_accuracy':rho,'p_value_descriptive':p})
confound_associations=pd.DataFrame(association_rows)

importance_stability=(fold_importance.groupby(['task','feature']).agg(
    top30_frequency=('feature','size'),median_rank=('rank','median'),median_importance=('importance','median'))
    .reset_index())
importance_stability['top30_fraction']=importance_stability.top30_frequency/(REPEATS*FOLDS)
importance_stability=(importance_stability.sort_values(['task','top30_fraction','median_importance'],
                                                       ascending=[True,False,False]))

save_table(recording_performance,'06_recording_level_oof_error_audit')
save_table(confound_associations,'07_confound_associations')
save_table(importance_stability,'08_feature_importance_stability')
display(confound_associations)
display(importance_stability.groupby('task').head(12))

### 5. Describe locked official-test errors and calibration

This section reads the already-locked P1-E08.1 models and test set. It does not select, refit or modify any estimator.

In [ ]:
def generalized_brier(y_true,probability,classes):
    onehot=np.column_stack([np.asarray(y_true)==label for label in classes]).astype(float)
    return float(np.mean(np.sum((probability-onehot)**2,axis=1)))
def expected_calibration_error(y_true,prediction,confidence,bins=10):
    correct=np.asarray(y_true)==np.asarray(prediction); confidence=np.asarray(confidence)
    edges=np.linspace(0,1,bins+1); value=0.0
    for low,high in zip(edges[:-1],edges[1:]):
        mask=(confidence>=low)&(confidence<(high if high<1 else high+1e-12))
        if mask.any(): value+=mask.mean()*abs(correct[mask].mean()-confidence[mask].mean())
    return float(value)

test_error_rows=[]; calibration_rows=[]; calibration_bin_rows=[]
for task,classes in TASKS.items():
    artifact=artifacts[task]; columns=np.asarray(artifact['feature_indices'],dtype=int)
    test_mask=metadata.experimental_label.isin(classes) & metadata.official_split.eq('test')
    indices=np.flatnonzero(test_mask.to_numpy()); estimator=artifact['estimator']
    prediction=estimator.predict(X[indices][:,columns])
    if not hasattr(estimator,'predict_proba'):
        probability=None; confidence=predict_confidence(estimator,X[indices][:,columns])
    else:
        raw_probability=estimator.predict_proba(X[indices][:,columns])
        model_classes=estimator.classes_.tolist()
        probability=np.column_stack([raw_probability[:,model_classes.index(label)] for label in classes])
        confidence=probability.max(axis=1)
    true=metadata.experimental_label.to_numpy()[indices]
    for global_index,true_label,predicted,conf in zip(indices,true,prediction,confidence):
        test_error_rows.append({'task':task,'recording_id':metadata.recording_id.iloc[global_index],
            'true_label':true_label,'predicted_label':predicted,'correct':bool(true_label==predicted),
            'confidence':float(conf),'age':metadata.age.iloc[global_index],
            'retained_seconds':metadata.retained_seconds.iloc[global_index],
            'selected_epochs':metadata.selected_epochs.iloc[global_index],
            'accepted_fraction':metadata.accepted_fraction.iloc[global_index]})
    ece=expected_calibration_error(true,prediction,confidence)
    calibration_rows.append({'task':task,'n_test':len(indices),'mean_confidence':float(np.mean(confidence)),
        'accuracy':float(accuracy_score(true,prediction)),'expected_calibration_error':ece,
        'generalized_brier_score':generalized_brier(true,probability,classes) if probability is not None else np.nan})
    edges=np.linspace(0,1,6)
    for low,high in zip(edges[:-1],edges[1:]):
        mask=(confidence>=low)&(confidence<(high if high<1 else high+1e-12))
        if mask.any(): calibration_bin_rows.append({'task':task,'confidence_low':low,'confidence_high':high,
            'n':int(mask.sum()),'mean_confidence':float(confidence[mask].mean()),
            'observed_accuracy':float((true[mask]==prediction[mask]).mean())})

test_error_audit=pd.DataFrame(test_error_rows)
calibration_summary=pd.DataFrame(calibration_rows)
calibration_bins=pd.DataFrame(calibration_bin_rows)
error_summary=(test_error_audit.groupby(['task','true_label','predicted_label']).size()
               .rename('recordings').reset_index())
save_table(test_error_audit,'09_LOCKED_TEST_ERROR_AUDIT')
save_table(error_summary,'10_LOCKED_TEST_CONFUSION_COUNTS')
save_table(calibration_summary,'11_LOCKED_TEST_CALIBRATION_SUMMARY')
save_table(calibration_bins,'12_LOCKED_TEST_CALIBRATION_BINS')
display(calibration_summary); display(error_summary)

### 6. Publication-ready stability and diagnostic figures

In [ ]:
fig,axes=plt.subplots(1,3)
for ax,(task,group) in zip(axes,repeat_metrics.groupby('task')):
    sns.boxplot(data=group,x='scenario',y='balanced_accuracy',hue='scenario',legend=False,ax=ax)
    ax.set(title=task.replace('_',' '),xlabel='',ylabel='Repeated-CV balanced accuracy')
    ax.tick_params(axis='x',rotation=35)
savefig(fig,'Figure_01_stability_and_confound_sensitivities',14,4.5); plt.show()

fig,axes=plt.subplots(1,3)
for ax,task in zip(axes,TASKS):
    use=importance_stability.loc[importance_stability.task.eq(task)].head(12).sort_values('top30_fraction')
    ax.barh(use.feature,use.top30_fraction,color='#56B4E9')
    ax.set(title=task.replace('_',' '),xlabel='Top-30 selection frequency',xlim=(0,1))
savefig(fig,'Figure_02_feature_stability',15,5); plt.show()

fig,axes=plt.subplots(1,3)
for ax,task in zip(axes,TASKS):
    use=calibration_bins.loc[calibration_bins.task.eq(task)]
    ax.plot([0,1],[0,1],'--',color='0.5')
    ax.plot(use.mean_confidence,use.observed_accuracy,'o-',color='#D55E00')
    ax.set(title=task.replace('_',' '),xlabel='Mean confidence',ylabel='Observed accuracy',xlim=(0,1),ylim=(0,1))
savefig(fig,'Figure_03_locked_test_calibration',12,3.8); plt.show()

fig,axes=plt.subplots(1,3)
for ax,(task,group) in zip(axes,recording_performance.groupby('task')):
    sns.scatterplot(data=group,x='age',y='oof_accuracy',hue='experimental_label',ax=ax,s=20,alpha=.7)
    ax.set(title=task.replace('_',' '),ylabel='Repeated OOF accuracy',xlabel='Age')
    ax.legend(frameon=False,fontsize=7)
savefig(fig,'Figure_04_age_and_oof_accuracy',13,4); plt.show()

## Checks

### 7. Final diagnostic integrity gates

In [ ]:
expected_predictions=0
for task,classes in TASKS.items():
    development=metadata.experimental_label.isin(classes) & metadata.official_split.isin(['train','validation'])
    expected_predictions+=int(development.sum())*REPEATS*3
    expected_predictions+=int((development & metadata.selected_epochs.eq(12)).sum())*REPEATS

coverage=(oof_predictions.groupby(['task','scenario','repeat']).recording_id.nunique().reset_index())
checks=pd.DataFrame([
 {'check':'P1-E08.1 benchmark authenticated','pass':input_checks['pass'].all(),'detail':'all input gates passed'},
 {'check':'all three tasks and four scenarios completed','pass':len(stability_summary)==len(TASKS)*4,'detail':len(stability_summary)},
 {'check':'ten repeats completed per task/scenario','pass':stability_summary.repeats.eq(REPEATS).all(),'detail':stability_summary.repeats.value_counts().to_dict()},
 {'check':'OOF prediction count reconciles','pass':len(oof_predictions)==expected_predictions,'detail':{'observed':len(oof_predictions),'expected':expected_predictions}},
 {'check':'each development recording predicted once per repeat/scenario','pass':not oof_predictions.duplicated(['task','scenario','repeat','recording_id']).any(),'detail':len(oof_predictions)},
 {'check':'age-only sensitivity completed','pass':set(stability_summary.loc[stability_summary.scenario.eq('age_only'),'task'])==set(TASKS),'detail':'three tasks'},
 {'check':'age-residualized sensitivity completed','pass':set(stability_summary.loc[stability_summary.scenario.eq('age_residualized_eeg'),'task'])==set(TASKS),'detail':'three tasks'},
 {'check':'equal-12-epoch sensitivity completed','pass':set(stability_summary.loc[stability_summary.scenario.eq('equal_12_epochs'),'task'])==set(TASKS),'detail':'three tasks'},
 {'check':'feature stability uses locked EEG folds only','pass':fold_importance.groupby('task').size().gt(0).all(),'detail':fold_importance.groupby('task').size().to_dict()},
 {'check':'locked test predictions reproduced without refitting','pass':len(test_error_audit)==len(locked_test_predictions) and test_error_audit.groupby('task').size().to_dict()==locked_test_predictions.groupby('task').size().to_dict(),'detail':len(test_error_audit)},
 {'check':'official test not used for selection or tuning','pass':True,'detail':'descriptive locked-model analysis only'},
 {'check':'sex confounding not falsely assessed','pass':True,'detail':'sex unavailable in released CAUEEG'},
 {'check':'analysis remains recording-level','pass':metadata.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':metadata.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~metadata.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':metadata.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(checks,'13_final_integrity_checks'); display(checks)
status=('P1-E09.1 PASSED — STABILITY, CONFOUND AND ERROR AUDIT LOCKED'
        if checks['pass'].all() else 'STOP — P1-E09.1 CHECKS FAILED; DO NOT FINALISE PAPER ONE BENCHMARK')

summary_csv=TABLES/'03_stability_and_sensitivity_summary.csv'
association_csv=TABLES/'07_confound_associations.csv'
calibration_csv=TABLES/'11_LOCKED_TEST_CALIBRATION_SUMMARY.csv'
run_record={'experiment':'P1-E09.1 CAUEEG benchmark stability, confound and error-analysis lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,
 'source_p1_e08_run_record':str(E08_RUN_RECORD),'source_feature_matrix_sha256':sha256_file(FEATURE_MATRIX),
 'repeats':REPEATS,'folds':FOLDS,'tasks':TASKS,
 'diagnostic_scenarios':['locked_eeg','age_only','age_residualized_eeg','equal_12_epochs'],
 'stability_summary_csv':str(summary_csv),'stability_summary_sha256':sha256_file(summary_csv),
 'confound_associations_csv':str(association_csv),'confound_associations_sha256':sha256_file(association_csv),
 'test_calibration_csv':str(calibration_csv),'test_calibration_sha256':sha256_file(calibration_csv),
 'test_role':'descriptive analysis of previously locked predictions; no selection, fitting or tuning',
 'sex_confound_assessment':'not possible because sex was removed from released CAUEEG',
 'identity_interpretation':{'analysis_unit':'recording','participant_identity_verified':False},
 'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all(): raise AssertionError(status)

## Next Steps

After a complete P1-E09.1 PASS:

1. Interpret stability, age-only performance, age-residualized performance and equal-epoch sensitivity before making claims about disease-specific EEG information.
2. Report sex confounding as unavailable, not as controlled.
3. Do not alter P1-E08. models using official test errors.
4. P1-E10.1 should consolidate the locked cohort, preprocessing, automatic-QA, conventional benchmark, and stability 
